## 1. Setup and load inputs

I load:
- Prices and returns (Notebook 01, 02)
- Covariance matrix $\Sigma$ (Notebook 02)
- Expected returns from MLR (Notebook 05), MA (Notebook 04), and historical realized returns

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

# Plot styling (consistent across notebooks)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('tab10')

np.random.seed(42)

# Annualization factor for HOSE
TRADING_DAYS = 250
RISK_FREE_RATE = 0.04  # Vietnam 10Y government bond ~4% (proxy)

# ============================================================
# Load all inputs from previous notebooks
# ============================================================

# 1. Prices and returns (NB01, NB02)
prices = pd.read_csv('../data/prices.csv', index_col='time', parse_dates=True)
returns = pd.read_csv('../data/returns.csv', index_col='time', parse_dates=True)

# 2. Covariance matrix (NB02) — annualized
cov_matrix = pd.read_csv('../data/cov_matrix.csv', index_col=0)

# 3. MLR-implied expected returns (NB05) — main input
mlr_mu = pd.read_csv('../data/mlr_expected_returns.csv', index_col=0)
mlr_mu = mlr_mu['mu_annualized']  # convert to Series

# 4. MA-implied expected returns (NB04) — baseline comparison
ma_mu = pd.read_csv('../data/ma_expected_returns.csv', index_col=0)
ma_mu = ma_mu.iloc[:, 0]  # take first column (annualized μ)

# 5. Historical realized returns (NB02) — sanity check
historical_mu = returns.mean() * TRADING_DAYS

# ============================================================
# Verify inputs
# ============================================================
TICKERS = prices.columns.tolist()
N_STOCKS = len(TICKERS)

print(f'Number of stocks:  {N_STOCKS}')
print(f'Tickers:           {TICKERS}')
print(f'Trading period:    {prices.index.min().date()} → {prices.index.max().date()}')
print(f'Risk-free rate:    {RISK_FREE_RATE*100:.1f}% (annualized)')

print(f'\n--- Expected returns comparison (annualized %) ---')
mu_comparison = pd.DataFrame({
    'Historical': historical_mu * 100,
    'MA5-implied': ma_mu * 100,
    'MLR-implied': mlr_mu * 100,
})
print(mu_comparison.round(2))

print(f'\n--- Covariance matrix Σ ---')
print(f'Shape: {cov_matrix.shape}')
print(cov_matrix.round(4))

# Sanity check: tickers align
assert all(cov_matrix.index == TICKERS), 'Σ tickers misaligned!'
assert all(mlr_mu.index == TICKERS), 'MLR μ tickers misaligned!'
print(f'\n✓ All inputs aligned on {N_STOCKS} tickers')

Number of stocks:  8
Tickers:           ['VNM', 'VIC', 'VHM', 'FPT', 'HPG', 'MWG', 'VCB', 'MBB']
Trading period:    2021-11-01 → 2025-04-29
Risk-free rate:    4.0% (annualized)

--- Expected returns comparison (annualized %) ---
     Historical  MA5-implied  MLR-implied
VNM       -5.68          8.0        20.92
VIC       -4.87         13.4        21.27
VHM       -4.75         11.3         7.05
FPT       22.39        -12.4        40.22
HPG       -5.76         14.6       -37.93
MWG        5.23          6.9       -24.94
VCB       11.41         -4.5       -20.16
MBB       13.75         -3.4       -30.97

--- Covariance matrix Σ ---
Shape: (8, 8)
        VNM     VIC     VHM     FPT     HPG     MWG     VCB     MBB
VNM  0.0002  0.0001  0.0001  0.0001  0.0001  0.0001  0.0001  0.0001
VIC  0.0001  0.0004  0.0002  0.0001  0.0001  0.0001  0.0001  0.0001
VHM  0.0001  0.0002  0.0004  0.0001  0.0002  0.0002  0.0001  0.0001
FPT  0.0001  0.0001  0.0001  0.0003  0.0002  0.0002  0.0001  0.0002
HPG  0.000

In [2]:
# ============================================================
# Implement core portfolio functions
# ============================================================
# Three building blocks:
#   1. Portfolio expected return:    μ_p = μ^T w
#   2. Portfolio volatility:         σ_p = sqrt(w^T Σ w)
#   3. Sharpe ratio:                 (μ_p - r_f) / σ_p
# ============================================================

def portfolio_return(weights, mu):
    """Compute portfolio expected return: μ_p = μ^T w.
    
    Linear algebra: matrix-vector product (1D × 1D = scalar).
    
    Parameters
    ----------
    weights : ndarray, shape (n,)
        Portfolio weights summing to 1.
    mu : ndarray, shape (n,)
        Expected return vector (annualized).
    
    Returns
    -------
    scalar : portfolio expected return (annualized).
    """
    return np.dot(mu, weights)

def portfolio_volatility(weights, sigma):
    """Compute portfolio volatility: σ_p = sqrt(w^T Σ w).
    
    Linear algebra: quadratic form (1D × 2D × 1D = scalar).
    Note: Σ is daily covariance, so I multiply by TRADING_DAYS
    to annualize the variance before taking sqrt.
    
    Parameters
    ----------
    weights : ndarray, shape (n,)
        Portfolio weights.
    sigma : ndarray, shape (n, n)
        Daily covariance matrix.
    
    Returns
    -------
    scalar : portfolio volatility (annualized).
    """
    daily_var = weights @ sigma @ weights  # quadratic form
    annual_var = daily_var * TRADING_DAYS
    return np.sqrt(annual_var)

def sharpe_ratio(weights, mu, sigma, rf=RISK_FREE_RATE):
    """Compute portfolio Sharpe ratio: (μ_p - r_f) / σ_p."""
    mu_p = portfolio_return(weights, mu)
    sigma_p = portfolio_volatility(weights, sigma)
    if sigma_p < 1e-10:  # avoid division by zero
        return 0.0
    return (mu_p - rf) / sigma_p

# ============================================================
# Sanity check: test on equal-weight portfolio
# ============================================================
print('--- Sanity check: equal-weight portfolio (1/N) ---')

w_equal = np.ones(N_STOCKS) / N_STOCKS  # all weights = 1/8 = 0.125
print(f'Equal weights: {w_equal}')
print(f'Sum of weights: {w_equal.sum():.4f} (should be 1.0)')

# Convert pandas inputs to numpy for the functions
mu_mlr_arr = mlr_mu.values
sigma_arr  = cov_matrix.values

ret_eq    = portfolio_return(w_equal, mu_mlr_arr)
vol_eq    = portfolio_volatility(w_equal, sigma_arr)
sharpe_eq = sharpe_ratio(w_equal, mu_mlr_arr, sigma_arr)

print(f'\nUsing MLR-implied μ:')
print(f'  Portfolio return     μ_p = {ret_eq*100:+.2f}%/year')
print(f'  Portfolio volatility σ_p = {vol_eq*100:.2f}%/year')
print(f'  Sharpe ratio              = {sharpe_eq:.4f}')

# Also compute for Historical and MA5 μ for comparison
print(f'\n--- Equal-weight portfolio with different μ estimates ---')
for name, mu_est in [('Historical', historical_mu.values),
                     ('MA5-implied', ma_mu.values),
                     ('MLR-implied', mu_mlr_arr)]:
    r = portfolio_return(w_equal, mu_est)
    s = sharpe_ratio(w_equal, mu_est, sigma_arr)
    print(f'  {name:15s}: return = {r*100:+6.2f}%,  Sharpe = {s:+.4f}')

--- Sanity check: equal-weight portfolio (1/N) ---
Equal weights: [0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125]
Sum of weights: 1.0000 (should be 1.0)

Using MLR-implied μ:
  Portfolio return     μ_p = -3.07%/year
  Portfolio volatility σ_p = 20.09%/year
  Sharpe ratio              = -0.3516

--- Equal-weight portfolio with different μ estimates ---
  Historical     : return =  +3.97%,  Sharpe = -0.0017
  MA5-implied    : return =  +4.24%,  Sharpe = +0.0118
  MLR-implied    : return =  -3.07%,  Sharpe = -0.3516


In [3]:
# ============================================================
# Objective function: negative Sharpe ratio
# ============================================================
# scipy.optimize.minimize is a minimizer, so to MAXIMIZE Sharpe,
# I minimize the NEGATIVE Sharpe ratio:
#
#   min  -Sharpe(w) = -(μ^T w - r_f) / sqrt(w^T Σ w)
#    w
# ============================================================

def neg_sharpe(weights, mu, sigma, rf=RISK_FREE_RATE):
    """Negative Sharpe ratio (to be MINIMIZED by scipy).
    
    Minimizing this is equivalent to maximizing the Sharpe ratio.
    
    Parameters
    ----------
    weights : ndarray, shape (n,)
        Portfolio weights.
    mu : ndarray, shape (n,)
        Expected return vector (annualized).
    sigma : ndarray, shape (n, n)
        Daily covariance matrix.
    rf : float
        Risk-free rate (annualized).
    
    Returns
    -------
    scalar : negative Sharpe ratio.
    """
    return -sharpe_ratio(weights, mu, sigma, rf)

# ============================================================
# Sanity check: evaluate at equal-weight portfolio
# ============================================================
print('--- Objective function evaluated at equal-weight portfolio ---')

mu_mlr_arr = mlr_mu.values
sigma_arr  = cov_matrix.values

ns = neg_sharpe(w_equal, mu_mlr_arr, sigma_arr)
print(f'  neg_sharpe(w_equal) = {ns:+.4f}')
print(f'  Expected:           = {-sharpe_eq:+.4f}  (= -Sharpe(w_equal))')

assert abs(ns - (-sharpe_eq)) < 1e-10, 'neg_sharpe mismatch!'
print(f'  ✓ neg_sharpe correctly returns the negative of Sharpe')

--- Objective function evaluated at equal-weight portfolio ---
  neg_sharpe(w_equal) = +0.3516
  Expected:           = +0.3516  (= -Sharpe(w_equal))
  ✓ neg_sharpe correctly returns the negative of Sharpe


## 4. Constraints and initial guess

I encode two practical investment constraints:

### Constraint 1 — Budget (fully invested)

$$\sum_{i=1}^{8} w_i = 1$$

The portfolio uses 100% of capital, with no cash position and no leverage. In scipy, this is an **equality constraint**: `np.sum(w) - 1 == 0`.

### Constraint 2 — Long-only

$$0 \leq w_i \leq 1 \quad \forall i$$

No short selling and no single-stock concentration above 100% (the latter follows automatically from the budget constraint when combined with non-negativity). In scipy, these are passed as **bounds** rather than constraints, which the SLSQP solver handles more efficiently.

In [4]:
# ============================================================
# Constraints + initial guess for optimization
# ============================================================
# Constraint 1 (equality): sum(w) = 1     → fully invested
# Constraint 2 (bounds):   0 <= w_i <= 1  → long-only, no leverage
# Initial guess: equal-weight (1/N) — feasible starting point
# ============================================================

# Budget constraint: sum(w) - 1 == 0
budget_constraint = {
    'type': 'eq',
    'fun':  lambda w: np.sum(w) - 1
}

# Long-only bounds: each w_i in [0, 1]
bounds = tuple((0.0, 1.0) for _ in range(N_STOCKS))

# Initial guess: equal-weight portfolio
w_init = np.ones(N_STOCKS) / N_STOCKS

# ============================================================
# Verify constraints on initial guess
# ============================================================
print('--- Verifying constraints on initial guess (equal-weight) ---')
print(f'Initial weights: {w_init}')
print(f'\nBudget constraint check:')
print(f'  sum(w_init) = {np.sum(w_init):.6f}')
print(f'  Constraint value (should be 0): {np.sum(w_init) - 1:.6e}')

print(f'\nLong-only check:')
all_nonneg = np.all(w_init >= 0)
all_leq1   = np.all(w_init <= 1)
print(f'  All w_i >= 0: {all_nonneg}')
print(f'  All w_i <= 1: {all_leq1}')

print(f'\nBounds applied to each weight:')
for i, t in enumerate(TICKERS):
    print(f'  {t}: w in [{bounds[i][0]}, {bounds[i][1]}]')

print(f'\n✓ Initial guess is feasible (satisfies both constraints)')

--- Verifying constraints on initial guess (equal-weight) ---
Initial weights: [0.125 0.125 0.125 0.125 0.125 0.125 0.125 0.125]

Budget constraint check:
  sum(w_init) = 1.000000
  Constraint value (should be 0): 0.000000e+00

Long-only check:
  All w_i >= 0: True
  All w_i <= 1: True

Bounds applied to each weight:
  VNM: w in [0.0, 1.0]
  VIC: w in [0.0, 1.0]
  VHM: w in [0.0, 1.0]
  FPT: w in [0.0, 1.0]
  HPG: w in [0.0, 1.0]
  MWG: w in [0.0, 1.0]
  VCB: w in [0.0, 1.0]
  MBB: w in [0.0, 1.0]

✓ Initial guess is feasible (satisfies both constraints)


In [5]:
# ============================================================
# Optimize portfolio: maximize Sharpe ratio
# ============================================================
# Method: SLSQP (Sequential Least Squares Programming)
# - Handles equality constraints (budget) + bounds (long-only)
# - Smooth gradient-based — fast for small problems
# - scipy.optimize default for this problem class
# ============================================================

def optimize_portfolio(mu, sigma, rf=RISK_FREE_RATE, verbose=True):
    """Find weights that maximize Sharpe ratio (long-only, fully invested).
    
    Parameters
    ----------
    mu : ndarray, shape (n,)
        Expected return vector (annualized).
    sigma : ndarray, shape (n, n)
        Daily covariance matrix.
    rf : float
        Risk-free rate (annualized).
    verbose : bool
        Print optimization details.
    
    Returns
    -------
    w_opt : ndarray, shape (n,)
        Optimal portfolio weights.
    result : scipy OptimizeResult object
        Full optimization output.
    """
    result = minimize(
        fun=neg_sharpe,
        x0=w_init,
        args=(mu, sigma, rf),
        method='SLSQP',
        bounds=bounds,
        constraints=[budget_constraint],
        options={'ftol': 1e-10, 'maxiter': 200, 'disp': verbose}
    )
    
    w_opt = result.x
    
    if verbose:
        print(f'  Converged: {result.success}')
        print(f'  Iterations: {result.nit}')
        print(f'  Optimal -Sharpe: {result.fun:+.4f}')
        print(f'  Optimal Sharpe:  {-result.fun:+.4f}')
    
    return w_opt, result

# ============================================================
# Run optimization for 3 different μ estimates
# ============================================================
historical_mu_arr = historical_mu.values
ma_mu_arr         = ma_mu.values
mlr_mu_arr        = mlr_mu.values

optimal_portfolios = {}

for name, mu_est in [('Historical', historical_mu_arr),
                     ('MA5',        ma_mu_arr),
                     ('MLR',        mlr_mu_arr)]:
    print(f'\n========== Optimizing with {name}-implied μ ==========')
    w_opt, result = optimize_portfolio(mu_est, sigma_arr, verbose=True)
    optimal_portfolios[name] = {
        'weights': w_opt,
        'mu':      mu_est,
        'result':  result,
    }

# ============================================================
# Display optimal weights
# ============================================================
print(f'\n\n========== Optimal portfolio weights ==========')
weights_df = pd.DataFrame({
    name: optimal_portfolios[name]['weights']
    for name in ['Historical', 'MA5', 'MLR']
}, index=TICKERS)

print((weights_df * 100).round(2).astype(str) + '%')

# ============================================================
# Verify each portfolio satisfies constraints
# ============================================================
print(f'\n========== Constraint verification ==========')
for name in ['Historical', 'MA5', 'MLR']:
    w = optimal_portfolios[name]['weights']
    s = w.sum()
    nz = (w > 1e-4).sum()  # count non-trivial positions
    print(f'  {name}: sum(w) = {s:.6f}, '
          f'non-zero positions = {nz}/{N_STOCKS}, '
          f'min(w) = {w.min():.4f}')


========== Optimizing with Historical-implied μ ==========
Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.6859514727140865
            Iterations: 7
            Function evaluations: 63
            Gradient evaluations: 7
  Converged: True
  Iterations: 7
  Optimal -Sharpe: -0.6860
  Optimal Sharpe:  +0.6860

========== Optimizing with MA5-implied μ ==========
Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.38107213955134633
            Iterations: 9
            Function evaluations: 81
            Gradient evaluations: 9
  Converged: True
  Iterations: 9
  Optimal -Sharpe: -0.3811
  Optimal Sharpe:  +0.3811

========== Optimizing with MLR-implied μ ==========
Optimization terminated successfully    (Exit mode 0)
            Current function value: -1.3797812681632888
            Iterations: 9
            Function evaluations: 81
            Gradient evaluations: 9
  Converged: True
  Iterations

## 6. Sparsity and concentrated portfolios

### Observation: the optimizer naturally produces sparse solutions

The portfolios from Section 5 contain only **2–3 non-zero positions** out of 8 candidate stocks, even without an explicit L1 sparsity penalty. This is not a coincidence; it reflects a mathematical property of the **long-only Sharpe-maximization problem**:

- The Sharpe ratio is **scale-invariant**: scaling all weights by a constant does not change Sharpe.
- The budget constraint $\sum_i w_i = 1$ + long-only bound $w_i \geq 0$ creates a **simplex** as the feasible region.
- The optimizer pushes capital toward the few stocks with the best risk-adjusted contribution, leaving the rest at the boundary ($w_i = 0$).

This is analogous to the **KKT (Karush-Kuhn-Tucker) conditions** for inequality-constrained optimization: at the optimum, either the bound is active ($w_i = 0$) or the marginal Sharpe contribution equates across all in-portfolio stocks.

In [1]:
# ============================================================
# Sparse portfolio variant (post-processing)
# ============================================================
# The optimizer already produces sparse solutions (2-3 stocks).
# This step cleans tiny numerical artifacts and demonstrates
# the L1-equivalent post-processing approach.
# ============================================================

SPARSITY_THRESHOLD = 0.01  # 1% — clean up rounding noise

def sparsify_weights(w, threshold=SPARSITY_THRESHOLD):
    """Zero-out small weights, then renormalize to sum = 1."""
    w_sparse = w.copy()
    w_sparse[w_sparse < threshold] = 0.0
    total = w_sparse.sum()
    if total > 0:
        w_sparse = w_sparse / total
    return w_sparse

# Apply to all 3 portfolios
sparse_portfolios = {}
for name in ['Historical', 'MA5', 'MLR']:
    w_dense = optimal_portfolios[name]['weights']
    w_sparse = sparsify_weights(w_dense)
    sparse_portfolios[name] = {
        'weights': w_sparse,
        'mu': optimal_portfolios[name]['mu'],
    }

# Display side-by-side
print('========== Dense vs Sparse weights (threshold = 1%) ==========')
comparison = pd.DataFrame(index=TICKERS)
for name in ['Historical', 'MA5', 'MLR']:
    comparison[f'{name}-dense']  = optimal_portfolios[name]['weights']
    comparison[f'{name}-sparse'] = sparse_portfolios[name]['weights']

print((comparison * 100).round(2).astype(str) + '%')

# Position count
print('\n========== Position count: dense vs sparse ==========')
for name in ['Historical', 'MA5', 'MLR']:
    nz_dense  = (optimal_portfolios[name]['weights'] > 1e-4).sum()
    nz_sparse = (sparse_portfolios[name]['weights'] > 1e-4).sum()
    print(f'  {name:12s}: dense = {nz_dense} positions, sparse = {nz_sparse} positions')

NameError: name 'optimal_portfolios' is not defined